<a href="https://colab.research.google.com/github/aycaaozturk/AML-project/blob/main/Combined_XGBoost_AML.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
from google.colab import drive

drive.mount("/content/drive")

Mounted at /content/drive


In [2]:
!pip install -q xgboost scikit-survival

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.0/4.0 MB 92.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.1/9.1 MB 130.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 222.1/222.1 kB 16.9 MB/s eta 0:00:00


In [3]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import xgboost as xgb

from sksurv.util import Surv
from sksurv.metrics import (
    concordance_index_censored,
    cumulative_dynamic_auc,
    integrated_brier_score
)

import time
import gc

In [4]:
# ============================================================
# PATHS
# ============================================================

BASE_DIR = Path(
    "/content/drive/MyDrive/Uniklinikum Würzburg/AML/"
    "Output Files 2/combined_survival_model_ready_wout_protocol/"
)

TRAIN_PATH = BASE_DIR / "combined_train_survival_model_ready.csv"
VALIDATION_PATH = BASE_DIR / "combined_validation_survival_model_ready.csv"
TEST_PATH = BASE_DIR / "combined_test_survival_model_ready.csv"


# ============================================================
# LOAD DATA
# ============================================================

train_df = pd.read_csv(TRAIN_PATH)
val_df   = pd.read_csv(VALIDATION_PATH)
test_df  = pd.read_csv(TEST_PATH)

print("Train:", train_df.shape)
print("Validation:", val_df.shape)
print("Test:", test_df.shape)

Train: (620, 68)
Validation: (177, 68)
Test: (89, 68)


In [5]:
# ============================================================
# COLUMN DEFINITIONS
# ============================================================

DURATION_COL = "OS_DAYS"
EVENT_COL = "OS_EVENT"

ID_COLS = [
    "PATIENT_ID",
    "SAMPLE_ID"
]

EXCLUDE_COLS = ID_COLS + [
    DURATION_COL,
    EVENT_COL
]

FEATURE_COLS = [
    col for col in train_df.columns
    if col not in EXCLUDE_COLS
]

print("Number of features:", len(FEATURE_COLS))

print("\nFeatures:")
for feature in FEATURE_COLS:
    print(feature)

Number of features: 64

Features:
AGE
BONE_MARROW_LEUKEMIC_BLAST_PERCENTAGE
PERIPHERAL_BLASTS_PERCENTAGE
WBC_LOG1P
SEX_Female
SEX_Male
RACE_American Indian or Alaska Native
RACE_Asian
RACE_Black or African American
RACE_Native Hawaiian or other Pacific Islander
RACE_Other
RACE_White
ETHNICITY_Hispanic or Latino
ETHNICITY_Not Hispanic or Latino
CNS_DISEASE_No
CNS_DISEASE_Yes
CHLOROMA_No
CHLOROMA_Yes
FAB_M0
FAB_M1
FAB_M2
FAB_M4
FAB_M5
FAB_M6
FAB_M7
FAB_NOS
PRIMARY_CYTOGENETIC_CODE_MLL
PRIMARY_CYTOGENETIC_CODE_Normal
PRIMARY_CYTOGENETIC_CODE_Other
PRIMARY_CYTOGENETIC_CODE_inv(16)
PRIMARY_CYTOGENETIC_CODE_t(8;21)
RISK_GROUP_CLEAN_High
RISK_GROUP_CLEAN_Low
RISK_GROUP_CLEAN_Missing
RISK_GROUP_CLEAN_Standard
CYTOGENETIC_COMPLEXITY_GROUP_0
CYTOGENETIC_COMPLEXITY_GROUP_1
CYTOGENETIC_COMPLEXITY_GROUP_2
CYTOGENETIC_COMPLEXITY_GROUP_3
CYTOGENETIC_COMPLEXITY_GROUP_4_plus
T_6_9
T_8_21
T_3_5_Q25_Q34
T_6_11_Q27_Q23
T_9_11_P22_Q23
T_10_11_P11_2_Q23
T_11_19_Q23_P13_1
INV_16
DEL_5Q
DEL_7Q
DEL_9Q
MONOSOMY

In [6]:
# ============================================================
# INPUT FEATURES
# ============================================================

X_train = train_df[FEATURE_COLS].astype(float)
X_val   = val_df[FEATURE_COLS].astype(float)
X_test  = test_df[FEATURE_COLS].astype(float)


# ============================================================
# SURVIVAL DATA
# ============================================================

durations_train = train_df[DURATION_COL].astype(float).values
events_train = train_df[EVENT_COL].astype(int).values

durations_val = val_df[DURATION_COL].astype(float).values
events_val = val_df[EVENT_COL].astype(int).values

durations_test = test_df[DURATION_COL].astype(float).values
events_test = test_df[EVENT_COL].astype(int).values


# XGBoost Cox labels:
# event     -> positive duration
# censored  -> negative duration

y_train_xgb = np.where(
    events_train == 1,
    durations_train,
    -durations_train
)

y_val_xgb = np.where(
    events_val == 1,
    durations_val,
    -durations_val
)

y_test_xgb = np.where(
    events_test == 1,
    durations_test,
    -durations_test
)

print("X_train:", X_train.shape)
print("X_val:", X_val.shape)
print("X_test:", X_test.shape)

X_train: (620, 64)
X_val: (177, 64)
X_test: (89, 64)


In [7]:
# ============================================================
# SCIKIT-SURVIVAL OBJECTS
# ============================================================

surv_train = Surv.from_arrays(
    event=events_train.astype(bool),
    time=durations_train
)

surv_val = Surv.from_arrays(
    event=events_val.astype(bool),
    time=durations_val
)

surv_test = Surv.from_arrays(
    event=events_test.astype(bool),
    time=durations_test
)

In [8]:
def calculate_cindex(events, durations, risk_scores):

    result = concordance_index_censored(
        events.astype(bool),
        durations,
        risk_scores
    )

    return result[0]

In [9]:
# ============================================================
# BRESLOW BASELINE CUMULATIVE HAZARD
# ============================================================

def estimate_breslow_baseline(
    durations,
    events,
    risk_scores
):

    durations = np.asarray(durations)
    events = np.asarray(events)
    risk_scores = np.asarray(risk_scores)

    # Event times only
    event_times = np.sort(
        np.unique(
            durations[events == 1]
        )
    )

    cumulative_hazard = []
    current_hazard = 0.0

    for t in event_times:

        # Number of events at time t
        d_t = np.sum(
            (durations == t) &
            (events == 1)
        )

        # Patients still at risk at time t
        at_risk = durations >= t

        risk_sum = np.sum(
            risk_scores[at_risk]
        )

        if risk_sum > 0:

            hazard_increment = (
                d_t / risk_sum
            )

            current_hazard += (
                hazard_increment
            )

        cumulative_hazard.append(
            current_hazard
        )

    return (
        event_times,
        np.asarray(cumulative_hazard)
    )

In [10]:
def predict_survival_probabilities(
    risk_scores,
    eval_times,
    baseline_times,
    baseline_cumhaz
):

    H0 = np.zeros(
        len(eval_times)
    )

    for i, t in enumerate(eval_times):

        valid = baseline_times <= t

        if np.any(valid):
            H0[i] = baseline_cumhaz[valid][-1]
        else:
            H0[i] = 0.0

    # shape:
    # patients × evaluation times

    survival_probs = np.exp(
        -np.outer(
            risk_scores,
            H0
        )
    )

    return survival_probs

In [11]:
# ============================================================
# TIME-DEPENDENT AUC
# ============================================================

AUC_TIMES = np.array(
    [
        365,     # 1 year
        730,     # 2 years
        1095     # 3 years
    ],
    dtype=float
)


def calculate_mean_auc(
    reference_surv,
    target_surv,
    target_durations,
    risk_scores
):

    # Keep only valid evaluation times
    max_allowed = min(
        np.max(durations_train),
        np.max(target_durations)
    )

    min_allowed = np.min(
        target_durations
    )

    valid_times = AUC_TIMES[
        (AUC_TIMES > min_allowed) &
        (AUC_TIMES < max_allowed)
    ]

    if len(valid_times) == 0:
        return np.nan

    try:

        auc_values, mean_auc = (
            cumulative_dynamic_auc(
                reference_surv,
                target_surv,
                risk_scores,
                valid_times
            )
        )

        return float(mean_auc)

    except Exception as e:

        print(
            "AUC warning:",
            str(e)
        )

        return np.nan

In [12]:
# ============================================================
# IBS
# ============================================================

def calculate_ibs(
    reference_surv,
    target_surv,
    target_durations,
    risk_scores,
    baseline_times,
    baseline_cumhaz
):

    lower = max(
        np.percentile(
            target_durations,
            5
        ),
        np.min(durations_train)
    )

    upper = min(
        np.percentile(
            target_durations,
            95
        ),
        np.max(durations_train)
    )

    eval_times = np.linspace(
        lower,
        upper,
        100
    )

    survival_probs = (
        predict_survival_probabilities(
            risk_scores,
            eval_times,
            baseline_times,
            baseline_cumhaz
        )
    )

    try:

        ibs = integrated_brier_score(
            reference_surv,
            target_surv,
            survival_probs,
            eval_times
        )

        return float(ibs)

    except Exception as e:

        print(
            "IBS warning:",
            str(e)
        )

        return np.nan

In [13]:
# ============================================================
# XGBOOST CONFIGURATIONS
# ============================================================

CONFIGS = [

    # --------------------------------------------------------
    # 1. Baseline
    # --------------------------------------------------------
    {
        "max_depth": 3,
        "learning_rate": 0.03,
        "min_child_weight": 1,
        "subsample": 0.8,
        "colsample_bytree": 0.8,
        "reg_lambda": 1.0,
        "reg_alpha": 0.0
    },

    # --------------------------------------------------------
    # Tree depth
    # --------------------------------------------------------
    {
        "max_depth": 2,
        "learning_rate": 0.03,
        "min_child_weight": 1,
        "subsample": 0.8,
        "colsample_bytree": 0.8,
        "reg_lambda": 1.0,
        "reg_alpha": 0.0
    },

    {
        "max_depth": 4,
        "learning_rate": 0.03,
        "min_child_weight": 1,
        "subsample": 0.8,
        "colsample_bytree": 0.8,
        "reg_lambda": 1.0,
        "reg_alpha": 0.0
    },

    {
        "max_depth": 5,
        "learning_rate": 0.03,
        "min_child_weight": 1,
        "subsample": 0.8,
        "colsample_bytree": 0.8,
        "reg_lambda": 1.0,
        "reg_alpha": 0.0
    },

    # --------------------------------------------------------
    # Learning rate
    # --------------------------------------------------------
    {
        "max_depth": 3,
        "learning_rate": 0.01,
        "min_child_weight": 1,
        "subsample": 0.8,
        "colsample_bytree": 0.8,
        "reg_lambda": 1.0,
        "reg_alpha": 0.0
    },

    {
        "max_depth": 3,
        "learning_rate": 0.05,
        "min_child_weight": 1,
        "subsample": 0.8,
        "colsample_bytree": 0.8,
        "reg_lambda": 1.0,
        "reg_alpha": 0.0
    },

    {
        "max_depth": 3,
        "learning_rate": 0.10,
        "min_child_weight": 1,
        "subsample": 0.8,
        "colsample_bytree": 0.8,
        "reg_lambda": 1.0,
        "reg_alpha": 0.0
    },

    # --------------------------------------------------------
    # Min child weight
    # --------------------------------------------------------
    {
        "max_depth": 3,
        "learning_rate": 0.03,
        "min_child_weight": 3,
        "subsample": 0.8,
        "colsample_bytree": 0.8,
        "reg_lambda": 1.0,
        "reg_alpha": 0.0
    },

    {
        "max_depth": 3,
        "learning_rate": 0.03,
        "min_child_weight": 5,
        "subsample": 0.8,
        "colsample_bytree": 0.8,
        "reg_lambda": 1.0,
        "reg_alpha": 0.0
    },

    # --------------------------------------------------------
    # Sampling
    # --------------------------------------------------------
    {
        "max_depth": 3,
        "learning_rate": 0.03,
        "min_child_weight": 1,
        "subsample": 0.6,
        "colsample_bytree": 0.8,
        "reg_lambda": 1.0,
        "reg_alpha": 0.0
    },

    {
        "max_depth": 3,
        "learning_rate": 0.03,
        "min_child_weight": 1,
        "subsample": 1.0,
        "colsample_bytree": 0.8,
        "reg_lambda": 1.0,
        "reg_alpha": 0.0
    },

    {
        "max_depth": 3,
        "learning_rate": 0.03,
        "min_child_weight": 1,
        "subsample": 0.8,
        "colsample_bytree": 0.6,
        "reg_lambda": 1.0,
        "reg_alpha": 0.0
    },

    {
        "max_depth": 3,
        "learning_rate": 0.03,
        "min_child_weight": 1,
        "subsample": 0.8,
        "colsample_bytree": 1.0,
        "reg_lambda": 1.0,
        "reg_alpha": 0.0
    },

    # --------------------------------------------------------
    # L2 regularization
    # --------------------------------------------------------
    {
        "max_depth": 3,
        "learning_rate": 0.03,
        "min_child_weight": 1,
        "subsample": 0.8,
        "colsample_bytree": 0.8,
        "reg_lambda": 5.0,
        "reg_alpha": 0.0
    },

    {
        "max_depth": 3,
        "learning_rate": 0.03,
        "min_child_weight": 1,
        "subsample": 0.8,
        "colsample_bytree": 0.8,
        "reg_lambda": 10.0,
        "reg_alpha": 0.0
    },

    # --------------------------------------------------------
    # L1 regularization
    # --------------------------------------------------------
    {
        "max_depth": 3,
        "learning_rate": 0.03,
        "min_child_weight": 1,
        "subsample": 0.8,
        "colsample_bytree": 0.8,
        "reg_lambda": 1.0,
        "reg_alpha": 0.1
    },

    {
        "max_depth": 3,
        "learning_rate": 0.03,
        "min_child_weight": 1,
        "subsample": 0.8,
        "colsample_bytree": 0.8,
        "reg_lambda": 1.0,
        "reg_alpha": 1.0
    }
]

print(
    "Number of configurations:",
    len(CONFIGS)
)

Number of configurations: 17


In [14]:
# ============================================================
# TRAIN + EVALUATE ONE CONFIG
# ============================================================

def train_xgb_config(
    config,
    config_id
):

    start_time = time.time()

    # --------------------------------------------------------
    # DMatrix
    # --------------------------------------------------------

    dtrain = xgb.DMatrix(
        X_train,
        label=y_train_xgb,
        feature_names=FEATURE_COLS
    )

    dval = xgb.DMatrix(
        X_val,
        label=y_val_xgb,
        feature_names=FEATURE_COLS
    )

    dtest = xgb.DMatrix(
        X_test,
        label=y_test_xgb,
        feature_names=FEATURE_COLS
    )

    # --------------------------------------------------------
    # Parameters
    # --------------------------------------------------------

    params = {

        "objective": "survival:cox",
        "eval_metric": "cox-nloglik",

        "tree_method": "hist",

        "max_depth":
            config["max_depth"],

        "eta":
            config["learning_rate"],

        "min_child_weight":
            config["min_child_weight"],

        "subsample":
            config["subsample"],

        "colsample_bytree":
            config["colsample_bytree"],

        "lambda":
            config["reg_lambda"],

        "alpha":
            config["reg_alpha"],

        "seed": 42,

        "verbosity": 0
    }

    # --------------------------------------------------------
    # Train
    # --------------------------------------------------------

    evals_result = {}

    model = xgb.train(

        params=params,

        dtrain=dtrain,

        num_boost_round=2000,

        evals=[
            (dtrain, "train"),
            (dval, "validation")
        ],

        early_stopping_rounds=50,

        evals_result=evals_result,

        verbose_eval=False
    )

    # --------------------------------------------------------
    # Predictions
    # --------------------------------------------------------

    train_risk = model.predict(
        dtrain,
        iteration_range=(
            0,
            model.best_iteration + 1
        )
    )

    val_risk = model.predict(
        dval,
        iteration_range=(
            0,
            model.best_iteration + 1
        )
    )

    test_risk = model.predict(
        dtest,
        iteration_range=(
            0,
            model.best_iteration + 1
        )
    )

    # ========================================================
    # C-INDEX
    # ========================================================

    train_cindex = calculate_cindex(
        events_train,
        durations_train,
        train_risk
    )

    val_cindex = calculate_cindex(
        events_val,
        durations_val,
        val_risk
    )

    test_cindex = calculate_cindex(
        events_test,
        durations_test,
        test_risk
    )

    # ========================================================
    # TIME-DEPENDENT AUC
    # ========================================================

    val_mean_auc = calculate_mean_auc(
        surv_train,
        surv_val,
        durations_val,
        val_risk
    )

    test_mean_auc = calculate_mean_auc(
        surv_train,
        surv_test,
        durations_test,
        test_risk
    )

    # ========================================================
    # BASELINE HAZARD
    # ========================================================

    baseline_times, baseline_cumhaz = (
        estimate_breslow_baseline(
            durations_train,
            events_train,
            train_risk
        )
    )

    # ========================================================
    # IBS
    # ========================================================

    val_ibs = calculate_ibs(
        surv_train,
        surv_val,
        durations_val,
        val_risk,
        baseline_times,
        baseline_cumhaz
    )

    test_ibs = calculate_ibs(
        surv_train,
        surv_test,
        durations_test,
        test_risk,
        baseline_times,
        baseline_cumhaz
    )

    # ========================================================
    # RESULT
    # ========================================================

    runtime = (
        time.time() - start_time
    )

    result = {

        "Config": config_id,

        "max_depth":
            config["max_depth"],

        "learning_rate":
            config["learning_rate"],

        "min_child_weight":
            config["min_child_weight"],

        "subsample":
            config["subsample"],

        "colsample_bytree":
            config["colsample_bytree"],

        "reg_lambda":
            config["reg_lambda"],

        "reg_alpha":
            config["reg_alpha"],

        "Best trees":
            model.best_iteration + 1,

        "Train C-index":
            train_cindex,

        "Val C-index":
            val_cindex,

        "Test C-index":
            test_cindex,

        "C-index gap":
            train_cindex - val_cindex,

        "Val Mean AUC":
            val_mean_auc,

        "Test Mean AUC":
            test_mean_auc,

        "Val IBS":
            val_ibs,

        "Test IBS":
            test_ibs,

        "Runtime sec":
            runtime
    }

    del model
    del dtrain
    del dval
    del dtest

    gc.collect()

    return result

In [15]:
# ============================================================
# HYPERPARAMETER LOOP
# ============================================================

results = []

for i, config in enumerate(
    CONFIGS,
    start=1
):

    print(
        f"\nTraining configuration "
        f"{i}/{len(CONFIGS)}"
    )

    print(config)

    try:

        result = train_xgb_config(
            config,
            i
        )

        results.append(result)

        print(
            f"Val C-index = "
            f"{result['Val C-index']:.4f} | "
            f"Test C-index = "
            f"{result['Test C-index']:.4f} | "
            f"Val AUC = "
            f"{result['Val Mean AUC']:.4f} | "
            f"Test AUC = "
            f"{result['Test Mean AUC']:.4f} | "
            f"Val IBS = "
            f"{result['Val IBS']:.4f} | "
            f"Test IBS = "
            f"{result['Test IBS']:.4f}"
        )

    except Exception as e:

        print(
            f"Configuration {i} encountered an error:"
        )

        print(
            f"{type(e).__name__}: {e}"
        )


Training configuration 1/17
{'max_depth': 3, 'learning_rate': 0.03, 'min_child_weight': 1, 'subsample': 0.8, 'colsample_bytree': 0.8, 'reg_lambda': 1.0, 'reg_alpha': 0.0}
Val C-index = 0.6637 | Test C-index = 0.6818 | Val AUC = 0.6820 | Test AUC = 0.7211 | Val IBS = 0.1938 | Test IBS = 0.1761

Training configuration 2/17
{'max_depth': 2, 'learning_rate': 0.03, 'min_child_weight': 1, 'subsample': 0.8, 'colsample_bytree': 0.8, 'reg_lambda': 1.0, 'reg_alpha': 0.0}
Val C-index = 0.6704 | Test C-index = 0.6704 | Val AUC = 0.6878 | Test AUC = 0.7231 | Val IBS = 0.1888 | Test IBS = 0.1762

Training configuration 3/17
{'max_depth': 4, 'learning_rate': 0.03, 'min_child_weight': 1, 'subsample': 0.8, 'colsample_bytree': 0.8, 'reg_lambda': 1.0, 'reg_alpha': 0.0}
Val C-index = 0.6748 | Test C-index = 0.6795 | Val AUC = 0.6931 | Test AUC = 0.7253 | Val IBS = 0.1905 | Test IBS = 0.1723

Training configuration 4/17
{'max_depth': 5, 'learning_rate': 0.03, 'min_child_weight': 1, 'subsample': 0.8, 'cols

In [16]:
# ============================================================
# RESULTS TABLE
# ============================================================

results_df = pd.DataFrame(
    results
)

# IMPORTANT:
# rank configurations using VALIDATION,
# not test performance
results_df = (
    results_df
    .sort_values(
        by="Val C-index",
        ascending=False
    )
    .reset_index(drop=True)
)

display(
    results_df.style.format({

        "learning_rate": "{:.3f}",

        "subsample": "{:.2f}",

        "colsample_bytree": "{:.2f}",

        "reg_lambda": "{:.2f}",

        "reg_alpha": "{:.2f}",

        "Train C-index": "{:.3f}",

        "Val C-index": "{:.3f}",

        "Test C-index": "{:.3f}",

        "C-index gap": "{:.3f}",

        "Val Mean AUC": "{:.3f}",

        "Test Mean AUC": "{:.3f}",

        "Val IBS": "{:.3f}",

        "Test IBS": "{:.3f}",

        "Runtime sec": "{:.1f}"
    })
)

,Config,max_depth,learning_rate,min_child_weight,subsample,colsample_bytree,reg_lambda,reg_alpha,Best trees,Train C-index,Val C-index,Test C-index,C-index gap,Val Mean AUC,Test Mean AUC,Val IBS,Test IBS,Runtime sec
0,4,5,0.030,1,0.80,0.80,1.00,0.00,59,0.855,0.681,0.687,0.174,0.701,0.723,0.193,0.173,0.2
1,8,3,0.030,3,0.80,0.80,1.00,0.00,133,0.793,0.679,0.661,0.114,0.689,0.710,0.187,0.178,0.2
2,17,3,0.030,1,0.80,0.80,1.00,1.00,81,0.772,0.678,0.677,0.093,0.693,0.722,0.190,0.176,0.2
3,15,3,0.030,1,0.80,0.80,10.00,0.00,133,0.775,0.677,0.675,0.098,0.691,0.723,0.191,0.177,0.2
4,10,3,0.030,1,0.60,0.80,1.00,0.00,121,0.806,0.676,0.665,0.130,0.686,0.709,0.190,0.180,0.2
5,7,3,0.100,1,0.80,0.80,1.00,0.00,34,0.793,0.675,0.644,0.118,0.692,0.686,0.192,0.180,0.1
6,9,3,0.030,5,0.80,0.80,1.00,0.00,130,0.785,0.675,0.673,0.110,0.684,0.727,0.190,0.175,0.2
7,3,4,0.030,1,0.80,0.80,1.00,0.00,77,0.829,0.675,0.680,0.154,0.693,0.725,0.191,0.172,2.9
8,14,3,0.030,1,0.80,0.80,5.00,0.00,81,0.765,0.674,0.686,0.091,0.688,0.736,0.192,0.177,0.2
9,13,3,0.030,1,0.80,1.00,1.00,0.00,133,0.808,0.674,0.646,0.134,0.689,0.690,0.190,0.183,0.2


In [ ]:
best_row = results_df.iloc[0]

print("=" * 60)
print("BEST VALIDATION CONFIGURATION")
print("=" * 60)

print(
    f"Config:             "
    f"{int(best_row['Config'])}"
)

print(
    f"Max depth:          "
    f"{int(best_row['max_depth'])}"
)

print(
    f"Learning rate:      "
    f"{best_row['learning_rate']}"
)

print(
    f"Min child weight:   "
    f"{best_row['min_child_weight']}"
)

print(
    f"Subsample:          "
    f"{best_row['subsample']}"
)

print(
    f"Colsample by tree:  "
    f"{best_row['colsample_bytree']}"
)

print(
    f"L2 lambda:          "
    f"{best_row['reg_lambda']}"
)

print(
    f"L1 alpha:           "
    f"{best_row['reg_alpha']}"
)

print(
    f"Best trees:         "
    f"{int(best_row['Best trees'])}"
)

print()

print(
    f"Validation C-index: "
    f"{best_row['Val C-index']:.3f}"
)

print(
    f"Test C-index:       "
    f"{best_row['Test C-index']:.3f}"
)

print(
    f"Validation AUC:     "
    f"{best_row['Val Mean AUC']:.3f}"
)

print(
    f"Test AUC:           "
    f"{best_row['Test Mean AUC']:.3f}"
)

print(
    f"Validation IBS:     "
    f"{best_row['Val IBS']:.3f}"
)

print(
    f"Test IBS:           "
    f"{best_row['Test IBS']:.3f}"
)